In [4]:
import sys
import subprocess
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "duckdb"
])
print("DuckDB installed successfully in the notebook environment.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 1.0 MB/s eta 0:00:00
DuckDB installed successfully in the notebook environment.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import duckdb
import pandas as pd
con = duckdb.connect()
print("DuckDB connection established successfully.")
print("DuckDB version:", duckdb.__version__)

DuckDB connection established successfully.
DuckDB version: 1.5.5


In [6]:
# Load the processed movie dataset

df = pd.read_csv("../data/processed/movies_feature_engineered.csv")
print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
display(df.head())

Dataset loaded successfully.
Dataset shape: (569655, 46)


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,genre_list,averageRating,numVotes,year,...,genre_sport,genre_talk_show,genre_thriller,genre_war,genre_western,runtime_category,rating_category,log_votes,popularity_category,rating_popularity_score
0,tt0000574,movie,The Story of the Kelly Gang,1906,70.0,"Action,Adventure,Biography","['Action', 'Adventure', 'Biography']",6.0,1084.0,1906,...,0,0,0,0,0,Short,Average,6.989335,Popular,41.936012
1,tt0000591,movie,The Prodigal Son,1907,90.0,Drama,['Drama'],4.8,40.0,1907,...,0,0,0,0,0,Medium,Low,3.713572,Low,17.825146
2,tt0000615,movie,Robbery Under Arms,1907,NaN,Drama,['Drama'],3.4,36.0,1907,...,0,0,0,0,0,Unknown,Low,3.610918,Low,12.277121
3,tt0000630,movie,Hamlet,1908,NaN,Drama,['Drama'],3.0,42.0,1908,...,0,0,0,0,0,Unknown,Low,3.761200,Low,11.283600
4,tt0000675,movie,Don Quijote,1908,NaN,Drama,['Drama'],3.8,32.0,1908,...,0,0,0,0,0,Unknown,Low,3.496508,Low,13.286729


In [7]:
print("Number of columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())

Number of columns: 46

Columns:
['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres', 'genre_list', 'averageRating', 'numVotes', 'year', 'decade', 'era', 'movie_age', 'genre_count', 'genre_action', 'genre_adult', 'genre_adventure', 'genre_animation', 'genre_biography', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_film_noir', 'genre_game_show', 'genre_history', 'genre_horror', 'genre_music', 'genre_musical', 'genre_mystery', 'genre_news', 'genre_reality_tv', 'genre_romance', 'genre_sci_fi', 'genre_sport', 'genre_talk_show', 'genre_thriller', 'genre_war', 'genre_western', 'runtime_category', 'rating_category', 'log_votes', 'popularity_category', 'rating_popularity_score']


In [8]:
# Register the Pandas DataFrame as a DuckDB table
con.register("movies", df)
print("Movie dataset registered as DuckDB table: movies")

Movie dataset registered as DuckDB table: movies


In [9]:
# Basic SQL query: count total movies
result = con.execute("""
    SELECT COUNT(*) AS total_movies
    FROM movies
""").fetchdf()

display(result)

,total_movies
0,569655


In [10]:
# Preview the movie dataset using SQL

result = con.execute("""
    SELECT *
    FROM movies
    LIMIT 10
""").fetchdf()

display(result)

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,genre_list,averageRating,numVotes,year,...,genre_sport,genre_talk_show,genre_thriller,genre_war,genre_western,runtime_category,rating_category,log_votes,popularity_category,rating_popularity_score
0,tt0000574,movie,The Story of the Kelly Gang,1906,70.0,"Action,Adventure,Biography","['Action', 'Adventure', 'Biography']",6.0,1084.0,1906,...,0,0,0,0,0,Short,Average,6.989335,Popular,41.936012
1,tt0000591,movie,The Prodigal Son,1907,90.0,Drama,['Drama'],4.8,40.0,1907,...,0,0,0,0,0,Medium,Low,3.713572,Low,17.825146
2,tt0000615,movie,Robbery Under Arms,1907,NaN,Drama,['Drama'],3.4,36.0,1907,...,0,0,0,0,0,Unknown,Low,3.610918,Low,12.277121
3,tt0000630,movie,Hamlet,1908,NaN,Drama,['Drama'],3.0,42.0,1908,...,0,0,0,0,0,Unknown,Low,3.761200,Low,11.283600
4,tt0000675,movie,Don Quijote,1908,NaN,Drama,['Drama'],3.8,32.0,1908,...,0,0,0,0,0,Unknown,Low,3.496508,Low,13.286729
5,tt0000679,movie,The Fairylogue and Radio-Plays,1908,120.0,"Adventure,Fantasy","['Adventure', 'Fantasy']",4.9,82.0,1908,...,0,0,0,0,0,Medium,Low,4.418841,Low,21.652319
6,tt0000886,movie,"Hamlet, Prince of Denmark",1910,NaN,Drama,['Drama'],4.0,49.0,1910,...,0,0,0,0,0,Unknown,Low,3.912023,Low,15.648092
7,tt0000941,movie,Locura de amor,1909,45.0,Drama,['Drama'],4.2,39.0,1909,...,0,0,0,0,0,Short,Low,3.688879,Low,15.493294
8,tt0001028,movie,Salome Mad,1909,NaN,Comedy,['Comedy'],3.6,31.0,1909,...,0,0,0,0,0,Unknown,Low,3.465736,Low,12.476649
9,tt0001049,movie,Gøngehøvdingen,1909,NaN,"Drama,War","['Drama', 'War']",3.8,27.0,1909,...,0,0,0,1,0,Unknown,Low,3.332205,Low,12.662377


In [12]:
# Genre-wise movie count using SQL

genre_counts = con.execute("""
    SELECT
        genres,
        COUNT(*) AS movie_count
    FROM movies
    WHERE genres IS NOT NULL
    GROUP BY genres
    ORDER BY movie_count DESC
    LIMIT 20
""").fetchdf()

display(genre_counts)

,genres,movie_count
0,Drama,116557
1,Documentary,107257
2,Comedy,42370
3,"Drama,Romance",13826
4,"Comedy,Drama",13695
5,Horror,12985
6,Action,11933
7,Thriller,9767
8,"Comedy,Romance",7303
9,Romance,7180


In [13]:
# Individual genre frequency using DuckDB SQL

individual_genres = con.execute("""
    SELECT
        TRIM(genre) AS genre,
        COUNT(*) AS movie_count
    FROM (
        SELECT UNNEST(STRING_SPLIT(genres, ',')) AS genre
        FROM movies
        WHERE genres IS NOT NULL
    )
    GROUP BY genre
    ORDER BY movie_count DESC
""").fetchdf()

display(individual_genres)

,genre,movie_count
0,Drama,240101
1,Documentary,139858
2,Comedy,109361
3,Action,49704
4,Romance,49304
5,Crime,38141
6,Thriller,37977
7,Horror,34077
8,Adventure,26762
9,Family,17923


In [14]:
# Average rating by individual genre using SQL

genre_ratings = con.execute("""
    SELECT
        TRIM(genre) AS genre,
        COUNT(*) AS movie_count,
        ROUND(AVG(averageRating), 2) AS avg_rating
    FROM (
        SELECT
            UNNEST(STRING_SPLIT(genres, ',')) AS genre,
            averageRating
        FROM movies
        WHERE genres IS NOT NULL
          AND averageRating IS NOT NULL
    )
    GROUP BY genre
    HAVING COUNT(*) >= 100
    ORDER BY avg_rating DESC
""").fetchdf()

display(genre_ratings)

,genre,movie_count,avg_rating
0,News,685,7.19
1,Documentary,57189,7.17
2,Biography,11075,6.91
3,Music,9430,6.80
4,History,9845,6.75
5,Sport,4492,6.63
6,Reality-TV,101,6.49
7,Film-Noir,876,6.46
8,War,6805,6.36
9,Animation,6507,6.26


In [ ]:
# Genre popularity across decades using SQL
genre_decade = con.execute("""
    SELECT
        year // 10 * 10 AS decade,
        TRIM(genre) AS genre,
        COUNT(*) AS movie_count
    FROM (
        SELECT
            year,
            UNNEST(STRING_SPLIT(genres, ',')) AS genre
        FROM movies
        WHERE genres IS NOT NULL
          AND year IS NOT NULL
    )
    GROUP BY decade, genre
    ORDER BY decade, movie_count DESC
""").fetchdf()

display(genre_decade.head(30))

,decade,genre,movie_count
0,1900,Documentary,71
1,1900,Drama,20
2,1900,Sport,9
3,1900,Comedy,6
4,1900,War,5
5,1900,Biography,4
6,1900,News,4
7,1900,History,3
8,1900,Adventure,3
9,1900,Musical,3


In [16]:
# Find the most popular genre in each decade

top_genre_by_decade = con.execute("""
    WITH genre_counts AS (
        SELECT
            year // 10 * 10 AS decade,
            TRIM(genre) AS genre,
            COUNT(*) AS movie_count
        FROM (
            SELECT
                year,
                UNNEST(STRING_SPLIT(genres, ',')) AS genre
            FROM movies
            WHERE genres IS NOT NULL
              AND year IS NOT NULL
        )
        GROUP BY decade, genre
    ),

    ranked_genres AS (
        SELECT
            decade,
            genre,
            movie_count,
            ROW_NUMBER() OVER (
                PARTITION BY decade
                ORDER BY movie_count DESC
            ) AS rank
        FROM genre_counts
    )

    SELECT
        decade,
        genre,
        movie_count
    FROM ranked_genres
    WHERE rank = 1
    ORDER BY decade
""").fetchdf()

display(top_genre_by_decade)

,decade,genre,movie_count
0,1900,Documentary,71
1,1910,Drama,6404
2,1920,Drama,10256
3,1930,Drama,9880
4,1940,Drama,7308
5,1950,Drama,11141
6,1960,Drama,13920
7,1970,Drama,15948
8,1980,Drama,17177
9,1990,Drama,16851


In [17]:
# Genre diversity across decades

genre_diversity = con.execute("""
    SELECT
        year // 10 * 10 AS decade,
        COUNT(DISTINCT TRIM(genre)) AS unique_genres
    FROM (
        SELECT
            year,
            UNNEST(STRING_SPLIT(genres, ',')) AS genre
        FROM movies
        WHERE genres IS NOT NULL
          AND year IS NOT NULL
    )
    GROUP BY decade
    ORDER BY decade
""").fetchdf()

display(genre_diversity)

,decade,unique_genres
0,1900,16
1,1910,22
2,1920,23
3,1930,23
4,1940,23
5,1950,23
6,1960,24
7,1970,25
8,1980,25
9,1990,25


In [19]:
# Average movie rating by decade
decade_ratings = con.execute("""
    SELECT
        FLOOR(year / 10) * 10 AS decade,
        COUNT(*) AS movie_count,
        ROUND(AVG(averageRating), 2) AS avg_rating
    FROM movies
    WHERE year IS NOT NULL
      AND averageRating IS NOT NULL
    GROUP BY FLOOR(year / 10) * 10
    ORDER BY decade
""").fetchdf()

display(decade_ratings)

,decade,movie_count,avg_rating
0,1900.0,106,3.27
1,1910.0,2012,5.71
2,1920.0,3957,5.83
3,1930.0,8990,6.01
4,1940.0,8888,6.14
5,1950.0,12855,6.15
6,1960.0,17127,6.04
7,1970.0,22732,5.89
8,1980.0,25128,5.95
9,1990.0,26102,6.02


In [20]:
# Average rating by genre using DuckDB SQL

genre_ratings = con.execute("""
    SELECT
        TRIM(genre) AS genre,
        COUNT(*) AS movie_count,
        ROUND(AVG(averageRating), 2) AS avg_rating
    FROM (
        SELECT
            averageRating,
            UNNEST(STRING_SPLIT(genres, ',')) AS genre
        FROM movies
        WHERE genres IS NOT NULL
          AND averageRating IS NOT NULL
    )
    GROUP BY TRIM(genre)
    HAVING COUNT(*) >= 100
    ORDER BY avg_rating DESC
""").fetchdf()

display(genre_ratings)

,genre,movie_count,avg_rating
0,News,685,7.19
1,Documentary,57189,7.17
2,Biography,11075,6.91
3,Music,9430,6.80
4,History,9845,6.75
5,Sport,4492,6.63
6,Reality-TV,101,6.49
7,Film-Noir,876,6.46
8,War,6805,6.36
9,Animation,6507,6.26


In [21]:
# Average audience votes by decade using DuckDB SQL

decade_votes = con.execute("""
    SELECT
        FLOOR(year / 10) * 10 AS decade,
        COUNT(*) AS movie_count,
        ROUND(AVG(numVotes), 2) AS avg_votes,
        MAX(numVotes) AS max_votes
    FROM movies
    WHERE year IS NOT NULL
      AND numVotes IS NOT NULL
    GROUP BY FLOOR(year / 10) * 10
    ORDER BY decade
""").fetchdf()

display(decade_votes)

,decade,movie_count,avg_votes,max_votes
0,1900.0,106,46.92,1084.0
1,1910.0,2012,143.64,28475.0
2,1920.0,3957,592.69,202647.0
3,1930.0,8990,816.93,471531.0
4,1940.0,8888,1294.94,659485.0
5,1950.0,12855,1465.19,998708.0
6,1960.0,17127,1494.17,898128.0
7,1970.0,22732,1893.81,2248906.0
8,1980.0,25128,3203.09,1519361.0
9,1990.0,26102,6630.45,3226746.0


In [22]:
# Genre performance summary using DuckDB SQL

genre_performance = con.execute("""
    SELECT
        TRIM(genre) AS genre,
        COUNT(*) AS movie_count,
        ROUND(AVG(averageRating), 2) AS avg_rating,
        ROUND(AVG(numVotes), 2) AS avg_votes
    FROM (
        SELECT
            averageRating,
            numVotes,
            UNNEST(STRING_SPLIT(genres, ',')) AS genre
        FROM movies
        WHERE genres IS NOT NULL
          AND averageRating IS NOT NULL
          AND numVotes IS NOT NULL
    )
    GROUP BY TRIM(genre)
    HAVING COUNT(*) >= 100
    ORDER BY avg_rating DESC
""").fetchdf()

display(genre_performance)

,genre,movie_count,avg_rating,avg_votes
0,News,685,7.19,367.13
1,Documentary,57189,7.17,318.80
2,Biography,11075,6.91,7453.10
3,Music,9430,6.80,2811.41
4,History,9845,6.75,3742.19
5,Sport,4492,6.63,4842.68
6,Reality-TV,101,6.49,86.56
7,Film-Noir,876,6.46,4779.54
8,War,6805,6.36,4340.27
9,Animation,6507,6.26,12003.24


In [23]:
# Top genres by audience engagement
top_genres_by_votes = con.execute("""
    SELECT
        TRIM(genre) AS genre,
        COUNT(*) AS movie_count,
        ROUND(AVG(numVotes), 2) AS avg_votes,
        ROUND(AVG(averageRating), 2) AS avg_rating
    FROM (
        SELECT
            numVotes,
            averageRating,
            UNNEST(STRING_SPLIT(genres, ',')) AS genre
        FROM movies
        WHERE genres IS NOT NULL
          AND numVotes IS NOT NULL
          AND averageRating IS NOT NULL
    )
    GROUP BY TRIM(genre)
    HAVING COUNT(*) >= 100
    ORDER BY avg_votes DESC
    LIMIT 10
""").fetchdf()

display(top_genres_by_votes)

,genre,movie_count,avg_votes,avg_rating
0,Sci-Fi,8207,19495.97,5.31
1,Adventure,20979,16111.44,5.84
2,Animation,6507,12003.24,6.26
3,Fantasy,10695,11513.12,5.82
4,Action,35656,10946.57,5.72
5,Mystery,13789,9936.63,5.79
6,Crime,30239,8328.28,5.96
7,Thriller,29796,7513.61,5.57
8,Biography,11075,7453.10,6.91
9,Horror,27061,5110.09,4.96


In [24]:
# Most popular genre in each decade
decade_genre_counts = con.execute("""
    WITH genre_counts AS (
        SELECT
            FLOOR(year / 10) * 10 AS decade,
            TRIM(genre) AS genre,
            COUNT(*) AS movie_count
        FROM (
            SELECT
                year,
                UNNEST(STRING_SPLIT(genres, ',')) AS genre
            FROM movies
            WHERE year IS NOT NULL
              AND genres IS NOT NULL
        )
        GROUP BY decade, TRIM(genre)
    ),

    ranked_genres AS (
        SELECT
            decade,
            genre,
            movie_count,
            ROW_NUMBER() OVER (
                PARTITION BY decade
                ORDER BY movie_count DESC
            ) AS rank
        FROM genre_counts
    )

    SELECT
        decade,
        genre,
        movie_count
    FROM ranked_genres
    WHERE rank = 1
    ORDER BY decade
""").fetchdf()

display(decade_genre_counts)

,decade,genre,movie_count
0,1900.0,Documentary,71
1,1910.0,Drama,6404
2,1920.0,Drama,10256
3,1930.0,Drama,9880
4,1940.0,Drama,7308
5,1950.0,Drama,11141
6,1960.0,Drama,13920
7,1970.0,Drama,15948
8,1980.0,Drama,17177
9,1990.0,Drama,16851


In [25]:
# SQL analysis summary

sql_summary = con.execute("""
    SELECT
        COUNT(*) AS total_movies,
        ROUND(AVG(averageRating), 2) AS overall_avg_rating,
        ROUND(AVG(numVotes), 2) AS overall_avg_votes,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year
    FROM movies
    WHERE averageRating IS NOT NULL
      AND numVotes IS NOT NULL
      AND year IS NOT NULL
""").fetchdf()

print("===== SQL ANALYSIS SUMMARY =====")
display(sql_summary)

===== SQL ANALYSIS SUMMARY =====


,total_movies,overall_avg_rating,overall_avg_votes,earliest_year,latest_year
0,337665,6.13,3822.31,1900,2026


In [26]:
# Save SQL analysis results

genre_counts.to_csv(
    "../data/processed/sql_genre_counts.csv",
    index=False
)

individual_genres.to_csv(
    "../data/processed/sql_individual_genre_counts.csv",
    index=False
)

genre_ratings.to_csv(
    "../data/processed/sql_genre_ratings.csv",
    index=False
)

genre_decade.to_csv(
    "../data/processed/sql_genre_decade.csv",
    index=False
)

top_genre_by_decade.to_csv(
    "../data/processed/sql_top_genre_by_decade.csv",
    index=False
)

decade_ratings.to_csv(
    "../data/processed/sql_decade_ratings.csv",
    index=False
)

decade_votes.to_csv(
    "../data/processed/sql_decade_votes.csv",
    index=False
)

genre_performance.to_csv(
    "../data/processed/sql_genre_performance.csv",
    index=False
)

top_genres_by_votes.to_csv(
    "../data/processed/sql_top_genres_by_votes.csv",
    index=False
)

decade_genre_counts.to_csv(
    "../data/processed/sql_decade_genre_counts.csv",
    index=False
)

sql_summary.to_csv(
    "../data/processed/sql_summary.csv",
    index=False
)

print("All SQL analysis results saved successfully.")

All SQL analysis results saved successfully.


# Phase 12: SQL Integration — Key Insights & Conclusion

## Key Insights

### 1. SQL-Based Genre Analysis

DuckDB was successfully integrated with Python and Pandas to perform
analytical queries on the feature-engineered movie dataset.

SQL queries were used to calculate movie counts, genre frequencies,
average ratings, audience engagement, and decade-wise trends.

### 2. Genre Popularity

The SQL analysis confirmed that Drama is the most frequently occurring
individual genre in the dataset, followed by other major genres such as
Documentary, Comedy, Action, and Romance.

### 3. Genre Evolution

Decade-wise SQL analysis demonstrated that genre popularity changes across
different periods. Drama remained a dominant genre across most decades,
while the presence of other genres varied over time.

### 4. Rating Analysis

Average movie ratings were calculated for individual genres and decades.
This provides a SQL-based validation of the rating trends explored during
the earlier exploratory analysis.

### 5. Audience Engagement

SQL aggregation was used to compare audience engagement across genres and
decades using IMDb vote counts.

### 6. Advanced SQL Techniques

The analysis used SQL features including:

- `SELECT`
- `WHERE`
- `GROUP BY`
- `ORDER BY`
- `COUNT`
- `AVG`
- `MAX`
- `COUNT(DISTINCT ...)`
- `STRING_SPLIT`
- `UNNEST`
- Common Table Expressions (CTEs)
- Window functions using `ROW_NUMBER()`

## Final Conclusion

Phase 12 successfully integrated DuckDB into the Movie Genre Evolution
Analyzer.

The feature-engineered movie dataset was registered as a DuckDB table and
analyzed using SQL queries. SQL was used to reproduce and extend several
important analytical findings from the previous exploratory and statistical
phases.

The integration demonstrates how SQL and Python can work together for
large-scale data analysis. DuckDB provides an efficient analytical SQL
engine while Pandas continues to provide flexible data manipulation and
result visualization capabilities.

Overall, the SQL integration adds a database-oriented analytical layer to
the project and strengthens the project's practical data engineering and
data analysis workflow.